# TCAV dynamique post-hoc pour l’approche Attention-Gated Recurrent Grad-CAM S3D

Ce notebook est dédié uniquement à l’analyse conceptuelle **TCAV** du modèle déjà entraîné par le notebook principal :

`attention-gated-reccurent-gradcam-s3d.ipynb`

Il ne contient pas d’entraînement du modèle principal. La logique est la suivante :

1. charger le checkpoint entraîné de l’approche Attention-Gated Recurrent Grad-CAM ;
2. reconstruire exactement la même architecture S3D + Spatial Gate + LSTM-Attention ;
3. créer ou charger la banque de concepts dynamiques ;
4. expliquer conceptuellement une vidéo ;
5. évaluer TCAV de manière conditionnée par classes pertinentes.

Les concepts dynamiques sont définis à partir de statistiques de mouvement extraites par optical flow. TCAV est utilisé ici comme une couche post-hoc complémentaire : il ne modifie pas la prédiction du modèle.

## 1. Imports globaux

Toutes les bibliothèques utilisées dans ce notebook sont importées ici afin de garder les cellules suivantes propres et lisibles.

In [24]:
# Imports globaux

import os
import cv2
import math
import json
import pickle
import random
import shutil
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

from torchvision import transforms
from torchvision.models.video import s3d, S3D_Weights, r3d_18, R3D_18_Weights

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from tqdm import tqdm
from IPython.display import Video, display, HTML

print("Torch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())


Torch: 2.10.0+cu128
CUDA disponible: True


## 2. Configuration minimale et reconstruction de l’approche étudiée

Cette section reconstruit uniquement les composants nécessaires pour l’inférence et l’analyse TCAV :

- dataset UCF101 ;
- modèle Attention-Gated Recurrent Grad-CAM ;
- Spatial Gate ;
- LSTM-Attention ;
- Grad-CAM vidéo ;
- fonctions d’affichage.

Les fonctions d’entraînement, les pertes XAI et les optimiseurs sont volontairement supprimés. Le modèle est chargé directement depuis un checkpoint déjà entraîné.

In [25]:
@dataclass
class CFG:
    """
    Configuration minimale pour l'analyse TCAV post-hoc.
    Cette configuration ne contient pas de paramètres d'entraînement.
    Les valeurs sauvegardées dans le checkpoint peuvent remplacer ces valeurs lors du chargement.
    """
    dataset_root: str = "/kaggle/input/datasets/matthewjansen/ucf101-action-recognition"
    backbone_name: str = "s3d"
    pretrained: bool = True
    num_classes: int = 101

    clip_len: int = 32
    frame_size: int = 224

    temporal_module: str = "lstm_attention"
    lstm_hidden_dim: int = 256
    lstm_bidirectional: bool = True
    transformer_layers: int = 1
    transformer_heads: int = 8
    transformer_dropout: float = 0.1
    temporal_hidden_dim: int = 512

    flow_size: int = 112
    flow_topk_ratio: float = 0.30

    seed: int = 42
    output_dir: str = "./xai_outputs_attention_gated_final"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def split_dir(cfg: CFG, split: str) -> str:
    return os.path.join(cfg.dataset_root, split)



# 2. Dataset UCF101 Folder Loader

class UCF101VideoFolderDataset(Dataset):
    """
    Organisation des dossiers:
        root_dir/class_name/video.avi
    """
    def __init__(
        self,
        root_dir: str,
        clip_len: int = 32,
        frame_size: int = 224,
        train: bool = False,
        class_to_idx: Optional[Dict[str, int]] = None,
        max_videos_per_class: Optional[int] = None,
    ):
        self.root_dir = root_dir
        self.clip_len = clip_len
        self.frame_size = frame_size
        self.train = train

        classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = class_to_idx if class_to_idx is not None else {c: i for i, c in enumerate(classes)}
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}

        exts = (".avi", ".mp4", ".mov", ".mkv")
        self.samples = []
        for class_name in classes:
            class_path = os.path.join(root_dir, class_name)
            videos = sorted([f for f in os.listdir(class_path) if f.lower().endswith(exts)])
            if max_videos_per_class is not None:
                videos = videos[:max_videos_per_class]
            if class_name not in self.class_to_idx:
                continue
            for f in videos:
                self.samples.append((os.path.join(class_path, f), self.class_to_idx[class_name], class_name))

        self.normalize = transforms.Normalize(
            mean=[0.43216, 0.394666, 0.37645],
            std=[0.22803, 0.22145, 0.216989],
        )

    def __len__(self):
        return len(self.samples)

    def _read_video(self, path: str) -> List[np.ndarray]:
        cap = cv2.VideoCapture(path)
        frames = []
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        cap.release()
        return frames

    def _sample_indices(self, n: int) -> np.ndarray:
        if n <= 0:
            return np.zeros(self.clip_len, dtype=np.int64)
        if n >= self.clip_len:
            if self.train:
                start = random.randint(0, max(0, n - self.clip_len))
                return np.arange(start, start + self.clip_len)
            return np.linspace(0, n - 1, self.clip_len).astype(np.int64)
        return np.linspace(0, n - 1, self.clip_len).astype(np.int64)

    def _preprocess(self, frames: List[np.ndarray]) -> Tuple[torch.Tensor, torch.Tensor]:
        idx = self._sample_indices(len(frames))
        sampled = []
        for i in idx:
            if len(frames) == 0:
                fr = np.zeros((self.frame_size, self.frame_size, 3), dtype=np.uint8)
            else:
                fr = frames[int(i)]
            fr = cv2.resize(fr, (self.frame_size, self.frame_size))
            if self.train and random.random() < 0.5:
                fr = cv2.flip(fr, 1)
            sampled.append(fr)

        arr = np.stack(sampled).astype(np.float32) / 255.0  # T,H,W,C

        raw = torch.from_numpy(arr).permute(3, 0, 1, 2).float()  # C,T,H,W

        x = raw.clone()
        for t in range(x.shape[1]):
            x[:, t] = self.normalize(x[:, t])

        return x, raw

    def __getitem__(self, idx: int):
        path, y, class_name = self.samples[idx]
        frames = self._read_video(path)
        x, raw = self._preprocess(frames)
        return {
            "video": x,
            "raw_video": raw,
            "label": torch.tensor(y, dtype=torch.long),
            "path": path,
            "class_name": class_name,
        }


def denormalize_video(x: torch.Tensor) -> np.ndarray:
    """
    x: [C,T,H,W] normalized or raw.
    retourne uint8 [T,H,W,C]
    """
    x = x.detach().cpu()
    if x.min() < 0:
        mean = torch.tensor([0.43216, 0.394666, 0.37645]).view(3, 1, 1, 1)
        std = torch.tensor([0.22803, 0.22145, 0.216989]).view(3, 1, 1, 1)
        x = x * std + mean
    x = x.clamp(0, 1)
    return (x.permute(1, 2, 3, 0).numpy() * 255).astype(np.uint8)


@torch.no_grad()
def compute_optical_flow_targets(
    video: torch.Tensor,
    out_hw: Optional[Tuple[int, int]] = None,
    flow_size: int = 112,
    topk_ratio: float = 0.30,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Calcule des pseudo-cibles de mouvement à partir de l'optical flow.

    vidéo d'entrée: [B,C,T,H,W], normalized or raw.
    retourne:
        temporal_flow: [B,T] normalized flow energy.
        spatial_flow: [B,1,T,Hout,Wout] normalized map in [0,1].
        binary_focus: [B,1,T,Hout,Wout] top-k pseudo object/motion region.
    """
    B, C, T, H, W = video.shape
    Hout, Wout = out_hw if out_hw is not None else (H, W)

    temporal_list = []
    spatial_list = []

    for b in range(B):
        frames = denormalize_video(video[b])
        gray_seq = []
        for fr in frames:
            gray = cv2.cvtColor(fr, cv2.COLOR_RGB2GRAY)
            gray = cv2.resize(gray, (flow_size, flow_size))
            gray_seq.append(gray)

        mags = []
        if T <= 1:
            mags = [np.zeros((flow_size, flow_size), dtype=np.float32)]
        else:
            first_mag = None
            for t in range(T - 1):
                prev = gray_seq[t]
                nxt = gray_seq[t + 1]
                flow = cv2.calcOpticalFlowFarneback(
                    prev, nxt, None,
                    pyr_scale=0.5, levels=3, winsize=15,
                    iterations=3, poly_n=5, poly_sigma=1.2, flags=0
                )
                mag = np.sqrt(flow[..., 0] ** 2 + flow[..., 1] ** 2).astype(np.float32)
                if first_mag is None:
                    first_mag = mag
                mags.append(mag)
            mags = [first_mag] + mags

        mags = np.stack(mags)  # T,h,w
        maps = []
        for t in range(T):
            m = mags[t]
            m = m / (m.max() + 1e-8)
            m = cv2.resize(m, (Wout, Hout))
            maps.append(m)
        maps = np.stack(maps).astype(np.float32)  # T,Hout,Wout

        temporal = maps.mean(axis=(1, 2))
        temporal = temporal / (temporal.sum() + 1e-8)

        temporal_list.append(temporal)
        spatial_list.append(maps)

    temporal_flow = torch.tensor(np.stack(temporal_list), dtype=torch.float32, device=video.device)
    spatial_flow = torch.tensor(np.stack(spatial_list), dtype=torch.float32, device=video.device).unsqueeze(1)

    flat = spatial_flow.flatten(2)
    k = max(1, int(flat.shape[-1] * topk_ratio))
    thresh = torch.topk(flat, k=k, dim=-1).values[..., -1].view(B, 1, 1, 1, 1)
    binary_focus = (spatial_flow >= thresh).float()

    return temporal_flow, spatial_flow, binary_focus


@torch.no_grad()
def compute_frame_diff_targets(
    video: torch.Tensor,
    out_hw: Optional[Tuple[int, int]] = None,
    topk_ratio: float = 0.30,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Fast fallback if optical flow is too slow.
    """
    B, C, T, H, W = video.shape
    Hout, Wout = out_hw if out_hw is not None else (H, W)
    raw = video.detach()
    if raw.min() < 0:
        mean = torch.tensor([0.43216, 0.394666, 0.37645], device=video.device).view(1, 3, 1, 1, 1)
        std = torch.tensor([0.22803, 0.22145, 0.216989], device=video.device).view(1, 3, 1, 1, 1)
        raw = (raw * std + mean).clamp(0, 1)
    diff = torch.abs(raw[:, :, 1:] - raw[:, :, :-1]).mean(dim=1, keepdim=True)
    first = diff[:, :, :1]
    maps = torch.cat([first, diff], dim=2)
    maps = F.interpolate(maps, size=(T, Hout, Wout), mode="trilinear", align_corners=False)
    maps = maps / (maps.amax(dim=(2, 3, 4), keepdim=True) + 1e-8)
    temporal = maps.mean(dim=(1, 3, 4))
    temporal = temporal / (temporal.sum(dim=1, keepdim=True) + 1e-8)

    flat = maps.flatten(2)
    k = max(1, int(flat.shape[-1] * topk_ratio))
    thresh = torch.topk(flat, k=k, dim=-1).values[..., -1].view(B, 1, 1, 1, 1)
    binary_focus = (maps >= thresh).float()
    return temporal, maps, binary_focus


def describe_dynamics_from_flow(temporal_attention: torch.Tensor, temporal_flow: torch.Tensor) -> str:
    """
    temporal_attention: [T] or [T_feature] normalized/sigmoid.
    temporal_flow: [T] normalized.
    """
    att = temporal_attention.detach().cpu().float()
    flow = temporal_flow.detach().cpu().float()

    if att.numel() != flow.numel():
        att = F.interpolate(att.view(1, 1, -1), size=flow.numel(), mode="linear", align_corners=False).view(-1)

    att_np = att.numpy()
    flow_np = flow.numpy()
    T = len(flow_np)

    important = np.where(att_np >= np.percentile(att_np, 75))[0]
    if len(important) == 0:
        important = np.array([int(np.argmax(att_np))])

    peak = int(np.argmax(att_np))
    span = f"Frames importantes approximatives: {int(important.min())} à {int(important.max())}, pic à la frame {peak}."

    flow_top = float(flow_np[important].mean())
    flow_all = float(flow_np.mean() + 1e-8)
    acceleration = np.abs(np.diff(flow_np))
    abruptness = float(acceleration.max()) if len(acceleration) else 0.0
    mean_acc = float(acceleration.mean() + 1e-8) if len(acceleration) else 0.0

    if flow_top > 1.35 * flow_all:
        dyn = "La décision est dominée par un segment à mouvement fort, ce qui indique une contribution de la vitesse ou de l'intensité de l'action."
    elif abruptness > 2.0 * mean_acc:
        dyn = "La décision est liée à une rupture dynamique, c'est-à-dire un changement brusque du mouvement entre deux instants."
    else:
        dyn = "La décision repose sur une dynamique continue et régulière plutôt que sur un changement brutal."

    return span + " | " + dyn



# 4. Model: Spatial Gate + Temporal Transformer Gate


class PositionalEncoding(nn.Module):
    def __init__(self, dim: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        pos = torch.arange(0, max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:pe[:, 1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x):
        return x + self.pe[:, :x.shape[1]]


class SpatialGate3D(nn.Module):
    """
    Apprend où regarder.
    Entrée: [B,C,T,H,W]
    Sortie: [B,1,T,H,W]
    """
    def __init__(self, in_channels: int):
        super().__init__()
        mid = max(in_channels // 8, 16)
        self.net = nn.Sequential(
            nn.Conv3d(in_channels, mid, kernel_size=1),
            nn.BatchNorm3d(mid),
            nn.ReLU(inplace=True),
            nn.Conv3d(mid, 1, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


class TemporalLSTMAttentionGate(nn.Module):
    """
    Learns WHEN to look using a recurrent temporal memory followed by attention.
    This is the default final variant because it keeps better classification accuracy
    while still providing explicit temporal importance.
    """
    def __init__(self, in_channels: int, hidden_dim: int = 256, bidirectional: bool = True, dropout: float = 0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=in_channels,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=bidirectional,
        )
        out_dim = hidden_dim * (2 if bidirectional else 1)
        self.att_head = nn.Sequential(
            nn.LayerNorm(out_dim),
            nn.Linear(out_dim, out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(out_dim // 2, 1),
        )
        self.context_proj = nn.Sequential(nn.LayerNorm(out_dim), nn.Linear(out_dim, out_dim), nn.Tanh())
        self.out_dim = out_dim

    def forward(self, frame_features: torch.Tensor):
        seq, _ = self.lstm(frame_features)
        logits = self.att_head(seq).squeeze(-1)
        beta = torch.sigmoid(logits)
        alpha = beta / (beta.sum(dim=1, keepdim=True) + 1e-8)
        context = torch.sum(seq * alpha.unsqueeze(-1), dim=1)
        context = self.context_proj(context)
        return context, alpha, beta, seq


class TemporalTransformerGate(nn.Module):
    """
    Learns WHEN to look.
    Replaces LSTM by Transformer Encoder.
    Uses sigmoid beta for coverage, and normalized alpha for context aggregation.
    """
    def __init__(self, in_channels: int, model_dim: int = 512, n_heads: int = 8, n_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.proj = nn.Linear(in_channels, model_dim)
        self.pos = PositionalEncoding(model_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=n_heads,
            dim_feedforward=model_dim * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.att_head = nn.Sequential(
            nn.LayerNorm(model_dim),
            nn.Linear(model_dim, model_dim // 2),
            nn.GELU(),
            nn.Linear(model_dim // 2, 1),
        )
        self.out_dim = model_dim

    def forward(self, frame_features: torch.Tensor):
        """
        frame_features: [B,T,C]
        retourne:
            context: [B,D]
            alpha: [B,T], sum=1, for weighted aggregation and Grad-CAM fusion
            beta: [B,T], sigmoid activation, for coverage and important frame detection
            seq: [B,T,D]
        """
        x = self.proj(frame_features)
        x = self.pos(x)
        seq = self.encoder(x)
        logits = self.att_head(seq).squeeze(-1)
        beta = torch.sigmoid(logits)
        alpha = beta / (beta.sum(dim=1, keepdim=True) + 1e-8)
        context = torch.sum(seq * alpha.unsqueeze(-1), dim=1)
        return context, alpha, beta, seq


class FinalAttentionGatedGradCAMModel(nn.Module):
    """
    Explainable-by-design video classifier.
    Backbone predicts action after gated spatio-temporal filtering.
    """
    def __init__(self, cfg: CFG, num_classes: int = 101):
        super().__init__()
        self.cfg = cfg
        self.backbone_name = cfg.backbone_name
        self.num_classes = num_classes

        if cfg.backbone_name == "s3d":
            weights = S3D_Weights.DEFAULT if cfg.pretrained else None
            base = s3d(weights=weights)
            self.features = base.features
            feat_dim = 1024
        elif cfg.backbone_name == "r3d_18":
            weights = R3D_18_Weights.DEFAULT if cfg.pretrained else None
            base = r3d_18(weights=weights)
            self.features = nn.Sequential(base.stem, base.layer1, base.layer2, base.layer3, base.layer4)
            feat_dim = 512
        else:
            raise ValueError("cfg.backbone_name must be 's3d' or 'r3d_18'")

        self.feat_dim = feat_dim
        self.spatial_gate = SpatialGate3D(feat_dim)
        if cfg.temporal_module == "lstm_attention":
            self.temporal_gate = TemporalLSTMAttentionGate(
                in_channels=feat_dim,
                hidden_dim=cfg.lstm_hidden_dim,
                bidirectional=cfg.lstm_bidirectional,
                dropout=cfg.transformer_dropout,
            )
        elif cfg.temporal_module == "transformer":
            self.temporal_gate = TemporalTransformerGate(
                in_channels=feat_dim,
                model_dim=cfg.temporal_hidden_dim,
                n_heads=cfg.transformer_heads,
                n_layers=cfg.transformer_layers,
                dropout=cfg.transformer_dropout,
            )
        else:
            raise ValueError("temporal_module must be 'lstm_attention' or 'transformer'")

        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(self.temporal_gate.out_dim, num_classes),
        )

    def forward(self, x: torch.Tensor, return_explanations: bool = True):
        feats = self.features(x)  # [B,C,T',H',W']
        spatial = self.spatial_gate(feats)
        gated = feats * spatial

        frame_features = gated.mean(dim=(3, 4)).permute(0, 2, 1)  # [B,T',C]
        context, alpha, beta, seq = self.temporal_gate(frame_features)
        logits = self.classifier(context)

        if not return_explanations:
            return logits

        return {
            "logits": logits,
            "features": feats,
            "gated_features": gated,
            "spatial_gate": spatial,
            "temporal_alpha": alpha,
            "temporal_beta": beta,
            "temporal_seq": seq,
        }




class VideoGradCAM:
    def __init__(self, model: nn.Module):
        self.model = model

    def __call__(self, video: torch.Tensor, class_idx: Optional[int] = None):
        """
        video: [1,C,T,H,W]
        retourne:
            heatmap [T,H,W]
            alpha [T_feature]
            beta [T_feature]
            pred_idx
            prob
        """
        self.model.train()
        video = video.clone().detach().to(next(self.model.parameters()).device)
        video.requires_grad_(True)

        out = self.model(video, return_explanations=True)
        logits = out["logits"]
        probs = torch.softmax(logits, dim=1)

        if class_idx is None:
            class_idx = int(probs.argmax(dim=1).item())

        score = logits[:, class_idx].sum()
        feats = out["gated_features"]

        self.model.zero_grad(set_to_none=True)
        grads = torch.autograd.grad(score, feats, retain_graph=True, create_graph=False)[0]

        weights = grads.mean(dim=(3, 4), keepdim=True)
        cam = F.relu((weights * feats).sum(dim=1, keepdim=True))

        spatial = out["spatial_gate"].detach()
        alpha = out["temporal_alpha"].detach()
        beta = out["temporal_beta"].detach()

        alpha_5d = alpha.view(1, 1, alpha.shape[1], 1, 1)
        if alpha.shape[1] != cam.shape[2]:
            alpha_5d = F.interpolate(alpha.view(1, 1, -1), size=cam.shape[2], mode="linear", align_corners=False).view(1, 1, cam.shape[2], 1, 1)

        fused = cam * spatial * alpha_5d
        fused = fused - fused.min()
        fused = fused / (fused.max() + 1e-8)

        fused_up = F.interpolate(
            fused,
            size=(video.shape[2], video.shape[3], video.shape[4]),
            mode="trilinear",
            align_corners=False,
        )
        heatmap = fused_up[0, 0].detach().cpu().numpy()
        return heatmap, alpha[0].detach().cpu(), beta[0].detach().cpu(), class_idx, float(probs[0, class_idx].detach().cpu())



# 7. Visualization
def overlay_heatmap(frame_rgb: np.ndarray, heat: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    heat_uint = np.uint8(255 * np.clip(heat, 0, 1))
    color = cv2.applyColorMap(heat_uint, cv2.COLORMAP_JET)
    color = cv2.cvtColor(color, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(frame_rgb, 1 - alpha, color, alpha, 0)


def save_explanation_video(
    video_tensor: torch.Tensor,
    heatmap: np.ndarray,
    temporal_beta: torch.Tensor,
    pred_text: str,
    prob: float,
    dynamics_text: str,
    save_path: str,
    fps: int = 8,
):
    frames = denormalize_video(video_tensor)
    T, H, W, _ = frames.shape

    beta = F.interpolate(temporal_beta.view(1, 1, -1), size=T, mode="linear", align_corners=False).view(-1).numpy()
    threshold = np.percentile(beta, 75)
    important = beta >= threshold

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(save_path, fourcc, fps, (W, H))

    for t in range(T):
        frame = overlay_heatmap(frames[t], heatmap[t])
        if important[t]:
            cv2.rectangle(frame, (3, 3), (W - 4, H - 4), (255, 0, 0), 5)

        txt1 = f"Prediction: {pred_text} ({prob:.2f})"
        txt2 = f"Temporal gate: {beta[t]:.3f}"
        cv2.putText(frame, txt1, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(frame, txt2, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 2, cv2.LINE_AA)
        writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

    writer.release()


def save_original_clip(sample, save_path: str, fps: int = 8):
    frames = denormalize_video(sample["video"])
    T, H, W, _ = frames.shape
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(save_path, fourcc, fps, (W, H))
    for fr in frames:
        writer.write(cv2.cvtColor(fr, cv2.COLOR_RGB2BGR))
    writer.release()
    return save_path


def convert_to_web_mp4(input_path: str, output_path: str):
    os.system(f'ffmpeg -y -i "{input_path}" -vcodec libx264 -pix_fmt yuv420p "{output_path}" -loglevel quiet')
    return output_path


def show_original_and_explanation(report, sample=None, width: int = 750):
    os.makedirs("./xai_outputs_v4/web", exist_ok=True)

    if sample is not None:
        original_path = "./xai_outputs_v4/web/original_tmp.mp4"
        save_original_clip(sample, original_path)
        original_web = "./xai_outputs_v4/web/original_web.mp4"
        convert_to_web_mp4(original_path, original_web)
        print("Vidéo originale")
        display(Video(original_web, embed=True, width=width))

    explained_web = "./xai_outputs_v4/web/explained_web.mp4"
    convert_to_web_mp4(report["output_video"], explained_web)
    print("Vidéo expliquée")
    display(Video(explained_web, embed=True, width=width))

    display(HTML(f"""
    <div style="font-family: Arial; max-width: 900px;">
      <h3>Explication complète</h3>
      <ul>
        <li><b>Ground truth:</b> {report.get("ground_truth")}</li>
        <li><b>Prediction:</b> {report.get("prediction")}</li>
        <li><b>Correct:</b> {report.get("correct")}</li>
        <li><b>Confidence:</b> {report.get("probability"):.4f}</li>
        <li><b>Dynamique:</b> {report.get("important_dynamics")}</li>
      </ul>
      <h4>Métriques</h4>
      <ul>
        <li><b>Deletion AUC:</b> {report.get("temporal_deletion_auc_lower_is_better"):.4f} ; plus bas = meilleure causalité.</li>
        <li><b>Insertion AUC:</b> {report.get("temporal_insertion_auc_higher_is_better"):.4f} ; plus haut = meilleure suffisance.</li>
        <li><b>Stabilité:</b> {report.get("temporal_stability_lower_is_better"):.6f} ; plus bas = moins de flickering.</li>
        <li><b>Motion proxy:</b> {report.get("motion_localization_proxy_higher_is_better"):.4f} ; plus haut = meilleur alignement dynamique.</li>
      </ul>
      <p>La heatmap indique les zones spatiales importantes. Le cadre rouge indique les frames sélectionnées par la porte temporelle.</p>
    </div>
    """))


# 8. Evaluation metrics
def find_available_checkpoint(path="./xai_outputs_attention_gated_final"):
    """
    Trouve automatiquement un checkpoint.
    Accepte soit un fichier .pt direct, soit un dossier contenant un checkpoint.
    """
    if os.path.isfile(path):
        print("Checkpoint utilisé:", path)
        return path

    candidates = [
        os.path.join(path, "best_attention_gated_final_model.pt"),
        os.path.join(path, "best_attention_gated_final_model.pth"),
        os.path.join(path, "last_checkpoint.pt"),
        os.path.join(path, "attention_reccurent_gradcam_s3d_model.pt"),
    ]

    for p in candidates:
        if os.path.exists(p):
            print("Checkpoint utilisé:", p)
            return p

    # fallback: prendre le premier .pt ou .pth trouvé dans le dossier
    if os.path.isdir(path):
        all_ckpts = []
        for root, _, files in os.walk(path):
            for f in files:
                if f.endswith((".pt", ".pth")):
                    all_ckpts.append(os.path.join(root, f))
        if len(all_ckpts) > 0:
            all_ckpts = sorted(all_ckpts)
            print("Checkpoint utilisé:", all_ckpts[0])
            return all_ckpts[0]

    raise FileNotFoundError(f"Aucun checkpoint trouvé dans {path}")






def load_model_from_checkpoint(checkpoint_path: str):
    cfg = CFG()
    ckpt = torch.load(checkpoint_path, map_location=cfg.device)
    if "cfg" in ckpt:
        for k, v in ckpt["cfg"].items():
            if hasattr(cfg, k):
                setattr(cfg, k, v)
    cfg.device = "cuda" if torch.cuda.is_available() else "cpu"

    model = FinalAttentionGatedGradCAMModel(cfg, num_classes=cfg.num_classes).to(cfg.device)
    model.load_state_dict(ckpt["model"], strict=True)
    model.eval()

    class_to_idx = ckpt.get("class_to_idx")
    return model, cfg, class_to_idx


def get_dataset_for_split(cfg: CFG, split: str, class_to_idx=None):
    return UCF101VideoFolderDataset(
        root_dir=split_dir(cfg, split),
        clip_len=cfg.clip_len,
        frame_size=cfg.frame_size,
        train=False,
        class_to_idx=class_to_idx,
    )


def explain_one_sample(checkpoint_path: str, sample_index: int = 0, split: str = "test", output_dir: Optional[str] = None):
    model, cfg, class_to_idx = load_model_from_checkpoint(checkpoint_path)
    dataset = get_dataset_for_split(cfg, split, class_to_idx)
    idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}

    output_dir = output_dir or cfg.output_dir
    os.makedirs(output_dir, exist_ok=True)

    sample = dataset[sample_index]
    video = sample["video"].unsqueeze(0).to(cfg.device)
    label = sample["label"].to(cfg.device)

    gradcam = VideoGradCAM(model)
    heatmap, alpha, beta, pred_idx, prob = gradcam(video)
    pred_text = idx_to_class.get(pred_idx, str(pred_idx))

    with torch.no_grad():
        temporal_flow, _, _ = compute_optical_flow_targets(video, out_hw=(cfg.frame_size, cfg.frame_size), flow_size=cfg.flow_size)
    dynamics = describe_dynamics_from_flow(
        F.interpolate(beta.view(1, 1, -1), size=cfg.clip_len, mode="linear", align_corners=False).view(-1),
        temporal_flow[0].detach().cpu(),
    )

    save_path = os.path.join(output_dir, f"explanation_{split}_{sample_index}_{pred_text}.mp4")
    save_explanation_video(sample["video"], heatmap, beta, pred_text, prob, dynamics, save_path)

    # Notebook TCAV post-hoc:
    # les métriques complètes de fidélité/stabilité/motion sont évaluées dans le notebook principal.
    # Ici, on garde uniquement l'explication vidéo + l'analyse conceptuelle TCAV.
    report = {
        "split": split,
        "sample_index": sample_index,
        "video_path": sample["path"],
        "ground_truth": sample["class_name"],
        "prediction": pred_text,
        "prediction_index": int(pred_idx),
        "correct": bool(pred_text == sample["class_name"]),
        "probability": float(prob),
        "important_dynamics": dynamics,
        "temporal_deletion_auc_lower_is_better": None,
        "temporal_insertion_auc_higher_is_better": None,
        "temporal_stability_lower_is_better": None,
        "motion_localization_proxy_higher_is_better": None,
        "output_video": save_path,
    }

    print(json.dumps(report, indent=2, ensure_ascii=False))
    return report, sample


def export_outputs(output_dir="./xai_outputs_v4"):
    zip_path = "/kaggle/working/xai_outputs_v4_export"
    shutil.make_archive(zip_path, "zip", output_dir)
    print("Archive prête:", zip_path + ".zip")

In [26]:
def load_attention_gated_model_for_tcav(checkpoint_path):
    """
    Charge le modèle Balanced déjà entraîné.
    Aucun ré-entraînement n'est effectué.
    """
    cfg = CFG()
    ckpt = torch.load(checkpoint_path, map_location=cfg.device)

    model = FinalAttentionGatedGradCAMModel(cfg, num_classes=cfg.num_classes).to(cfg.device)

    state_dict = ckpt["model"] if "model" in ckpt else ckpt.get("model_state_dict", ckpt)
    model.load_state_dict(state_dict, strict=True)
    model.eval()

    return model, cfg



## 3. Chargement du modèle Attention-Gated Recurrent Grad-CAM déjà entraîné

In [27]:
# Modifier ce chemin selon votre input Kaggle.
# Il peut pointer vers un dossier ou directement vers un fichier .pt.
CHECKPOINT_SOURCE = "/kaggle/input/models/younessouarda/attention-reccurent-gradcam-s3d-model/pytorch/default/1"

checkpoint_path = find_available_checkpoint(CHECKPOINT_SOURCE)

model_tcav, cfg_tcav = load_attention_gated_model_for_tcav(checkpoint_path)

print("Backbone:", cfg_tcav.backbone_name)
print("Module temporel:", cfg_tcav.temporal_module)
print("clip_len:", cfg_tcav.clip_len, "| frame_size:", cfg_tcav.frame_size)
print("Checkpoint chargé:", checkpoint_path)

# Vérification : ce notebook est dédié à l’approche proposée.
assert cfg_tcav.temporal_module == "lstm_attention", "Le checkpoint ne correspond pas à la version LSTM-Attention attendue."
assert cfg_tcav.backbone_name == "s3d", "Ce notebook est la version TCAV dédiée au backbone S3D."


Checkpoint utilisé: /kaggle/input/models/younessouarda/attention-reccurent-gradcam-s3d-model/pytorch/default/1/attention_reccurent_gradcam_s3d_model.pt
Backbone: s3d
Module temporel: lstm_attention
clip_len: 32 | frame_size: 224
Checkpoint chargé: /kaggle/input/models/younessouarda/attention-reccurent-gradcam-s3d-model/pytorch/default/1/attention_reccurent_gradcam_s3d_model.pt


## 4. Concepts dynamiques TCAV retenus

In [28]:
TCAV_DYNAMIC_CONCEPTS = [
    "mouvement_brusque",
    "changement_rapide",
    "mouvement_intermitent",
    "mouvement_accelere",
    "mouvement_periodique",
    "mouvement_localise",
    "mouvement_global",
    "mouvement_fin_precision",
]

CONCEPT_DESCRIPTIONS = {
    "mouvement_brusque": "forte rupture dynamique entre deux instants successifs",
    "changement_rapide": "grande variabilité du mouvement au cours du clip",
    "mouvement_intermitent": "mouvement concentré sur quelques instants, séparés par des phases calmes",
    "mouvement_accelere": "augmentation rapide de l'intensité du mouvement",
    "mouvement_periodique": "mouvement répétitif ou cyclique",
    "mouvement_localise": "mouvement concentré dans une petite zone spatiale",
    "mouvement_global": "mouvement réparti sur une grande partie de la scène",
    "mouvement_fin_precision": "mouvement faible, localisé et précis",
}

print("Concepts TCAV dynamiques retenus:")
for c in TCAV_DYNAMIC_CONCEPTS:
    print("-", c, ":", CONCEPT_DESCRIPTIONS[c])


Concepts TCAV dynamiques retenus:
- mouvement_brusque : forte rupture dynamique entre deux instants successifs
- changement_rapide : grande variabilité du mouvement au cours du clip
- mouvement_intermitent : mouvement concentré sur quelques instants, séparés par des phases calmes
- mouvement_accelere : augmentation rapide de l'intensité du mouvement
- mouvement_periodique : mouvement répétitif ou cyclique
- mouvement_localise : mouvement concentré dans une petite zone spatiale
- mouvement_global : mouvement réparti sur une grande partie de la scène
- mouvement_fin_precision : mouvement faible, localisé et précis


## 5. Extraction des statistiques dynamiques par optical flow

In [29]:
def tensor_video_to_uint8_numpy(video_tensor):
    """
    Convertit une vidéo tensor [C,T,H,W] vers numpy [T,H,W,3] uint8.
    Si denormalize_video est déjà défini dans le notebook, il est utilisé.
    """
    x = video_tensor.detach().cpu()

    if x.ndim != 4:
        raise ValueError("video_tensor doit être de forme [C,T,H,W]")

    if "denormalize_video" in globals():
        return denormalize_video(x)

    x = x.clamp(0, 1)
    arr = x.permute(1, 2, 3, 0).numpy()
    return (arr * 255).astype(np.uint8)


def compute_farneback_flow_sequence(video_np):
    """
    Calcule l'optical flow dense entre frames successives.

    Entrée:
        video_np: [T,H,W,3] uint8 RGB

    Sortie:
        flows: [T-1,H,W,2]
    """
    flows = []

    for t in range(len(video_np) - 1):
        f1 = cv2.cvtColor(video_np[t], cv2.COLOR_RGB2GRAY)
        f2 = cv2.cvtColor(video_np[t + 1], cv2.COLOR_RGB2GRAY)

        flow = cv2.calcOpticalFlowFarneback(
            f1, f2, None,
            pyr_scale=0.5,
            levels=3,
            winsize=15,
            iterations=3,
            poly_n=5,
            poly_sigma=1.2,
            flags=0
        )
        flows.append(flow)

    if len(flows) == 0:
        h, w = video_np.shape[1], video_np.shape[2]
        return np.zeros((1, h, w, 2), dtype=np.float32)

    return np.stack(flows).astype(np.float32)


def extract_dynamic_statistics(video_tensor):
    """
    Extrait des statistiques de mouvement à partir du flow.
    Ces statistiques servent à définir automatiquement les concepts dynamiques.
    """
    video_np = tensor_video_to_uint8_numpy(video_tensor)
    flows = compute_farneback_flow_sequence(video_np)

    u = flows[..., 0]
    v = flows[..., 1]
    magnitude = np.sqrt(u ** 2 + v ** 2)  # [T-1,H,W]

    frame_motion = magnitude.mean(axis=(1, 2))

    motion_mean = float(frame_motion.mean())
    motion_std = float(frame_motion.std())
    motion_max = float(frame_motion.max())

    if len(frame_motion) > 1:
        acceleration = np.abs(np.diff(frame_motion))
        acceleration_mean = float(acceleration.mean())
        acceleration_max = float(acceleration.max())
    else:
        acceleration_mean = 0.0
        acceleration_max = 0.0

    threshold_active = motion_mean + 0.5 * motion_std
    active_frames = frame_motion > threshold_active
    active_ratio = float(active_frames.mean())

    if len(active_frames) > 1:
        intermittence = float(np.mean(active_frames[1:] != active_frames[:-1]))
    else:
        intermittence = 0.0

    signal = frame_motion - frame_motion.mean()
    if np.abs(signal).sum() > 1e-8 and len(signal) > 3:
        ac = np.correlate(signal, signal, mode="full")
        ac = ac[len(ac) // 2:]
        periodicity_score = float(np.max(ac[1:]) / (ac[0] + 1e-8))
    else:
        periodicity_score = 0.0

    mag_mean = magnitude.mean(axis=0)
    flat = mag_mean.reshape(-1)

    if flat.max() > 1e-8:
        sorted_flat = np.sort(flat)[::-1]
        k = max(1, int(0.10 * len(sorted_flat)))
        localization_score = float(sorted_flat[:k].sum() / (sorted_flat.sum() + 1e-8))
    else:
        localization_score = 0.0

    global_threshold = float(mag_mean.mean() + 0.5 * mag_mean.std())
    global_motion_score = float((mag_mean > global_threshold).mean())

    fine_motion_score = float(localization_score * (1.0 / (1.0 + motion_mean)))

    return {
        "motion_mean": motion_mean,
        "motion_std": motion_std,
        "motion_max": motion_max,
        "acceleration_mean": acceleration_mean,
        "acceleration_max": acceleration_max,
        "active_ratio": active_ratio,
        "intermittence": intermittence,
        "periodicity_score": periodicity_score,
        "localization_score": localization_score,
        "global_motion_score": global_motion_score,
        "fine_motion_score": fine_motion_score,
        "frame_motion": frame_motion,
        "magnitude": magnitude,
    }


## 6. Seuils dynamiques et attribution automatique des concepts

In [30]:
def compute_dataset_dynamic_thresholds(dataset, max_samples=800):
    """
    Calcule des seuils dynamiques robustes sur un sous-ensemble du train set.
    """
    stats_list = []
    n = min(max_samples, len(dataset))

    for i in tqdm(range(n), desc="Scan dynamique pour seuils"):
        sample = dataset[i]
        stats = extract_dynamic_statistics(sample["video"])
        stats_list.append(stats)

    keys = [
        "motion_mean",
        "motion_std",
        "motion_max",
        "acceleration_mean",
        "acceleration_max",
        "active_ratio",
        "intermittence",
        "periodicity_score",
        "localization_score",
        "global_motion_score",
        "fine_motion_score",
    ]

    thresholds = {}
    for k in keys:
        values = np.array([s[k] for s in stats_list], dtype=np.float32)
        thresholds[k] = {
            "q25": float(np.quantile(values, 0.25)),
            "q50": float(np.quantile(values, 0.50)),
            "q60": float(np.quantile(values, 0.60)),
            "q70": float(np.quantile(values, 0.70)),
            "q75": float(np.quantile(values, 0.75)),
            "q80": float(np.quantile(values, 0.80)),
            "q90": float(np.quantile(values, 0.90)),
        }

    return thresholds


def assign_dynamic_concepts(stats, thresholds):
    """
    Attribue les concepts dynamiques à une vidéo selon ses statistiques de mouvement.
    """
    concepts = {}

    concepts["mouvement_brusque"] = int(
        stats["acceleration_max"] >= thresholds["acceleration_max"]["q80"]
    )

    concepts["changement_rapide"] = int(
        stats["motion_std"] >= thresholds["motion_std"]["q80"]
    )

    concepts["mouvement_intermitent"] = int(
        stats["intermittence"] >= thresholds["intermittence"]["q75"]
        and stats["active_ratio"] < thresholds["active_ratio"]["q75"]
    )

    concepts["mouvement_accelere"] = int(
        stats["acceleration_mean"] >= thresholds["acceleration_mean"]["q75"]
    )

    concepts["mouvement_periodique"] = int(
        stats["periodicity_score"] >= thresholds["periodicity_score"]["q80"]
    )

    concepts["mouvement_localise"] = int(
        stats["localization_score"] >= thresholds["localization_score"]["q75"]
    )

    concepts["mouvement_global"] = int(
        stats["global_motion_score"] >= thresholds["global_motion_score"]["q75"]
    )

    concepts["mouvement_fin_precision"] = int(
        stats["fine_motion_score"] >= thresholds["fine_motion_score"]["q75"]
        and stats["motion_mean"] <= thresholds["motion_mean"]["q60"]
    )

    return concepts



## 7. Extraction des activations de l’approche et sensibilité conceptuelle

In [31]:
@torch.no_grad()
def extract_tcav_activation(model, video, device):
    """
    Extrait un vecteur latent global [D] pour TCAV.
    """
    model.eval()
    video = video.unsqueeze(0).to(device)

    outputs = model(video, return_explanations=True)

    if "gated_features" in outputs:
        feats = outputs["gated_features"]
    elif "features" in outputs:
        feats = outputs["features"]
    else:
        raise KeyError("Le modèle doit retourner 'features' ou 'gated_features'.")

    # Pooling latent amélioré pour TCAV dynamique:
    # au lieu d'écraser complètement l'axe temporel par une moyenne,
    # on conserve le pic temporel le plus informatif.
    # feats: [B,C,T,H,W]
    temporal_feats = feats.mean(dim=(3, 4))      # [1,C,T]
    pooled = temporal_feats.max(dim=2)[0]       # [1,C]
    return pooled.squeeze(0).detach().cpu().numpy()


def extract_tcav_gradient_sensitivity(model, video, cav, device, class_idx=None):
    """
    Calcule une sensibilité TCAV approximative:
    gradient du score de classe par rapport aux features, projeté sur le CAV.
    
    Valeur positive: le concept pousse la prédiction vers la classe cible.
    Valeur négative: le concept va contre la classe cible.
    """
    model.eval()

    # Pour éviter certains soucis cuDNN avec LSTM backward en mode eval.
    previous_lstm_mode = None
    if hasattr(model, "temporal_gate") and hasattr(model.temporal_gate, "lstm"):
        previous_lstm_mode = model.temporal_gate.lstm.training
        model.temporal_gate.lstm.train()

    video = video.unsqueeze(0).to(device)
    video.requires_grad_(True)

    outputs = model(video, return_explanations=True)
    logits = outputs["logits"]

    if class_idx is None:
        class_idx = int(torch.softmax(logits, dim=1).argmax(dim=1).item())

    score = logits[:, class_idx].sum()
    feats = outputs["gated_features"] if "gated_features" in outputs else outputs["features"]

    model.zero_grad(set_to_none=True)
    with torch.backends.cudnn.flags(enabled=False):
        grads = torch.autograd.grad(score, feats, retain_graph=False, create_graph=False)[0]

    # Même logique que pour les activations TCAV:
    # moyenne spatiale puis agrégation max temporelle pour garder le moment le plus sensible.
    temporal_grads = grads.mean(dim=(3, 4))     # [1,C,T]
    pooled_grads = temporal_grads.max(dim=2)[0].squeeze(0).detach().cpu().numpy()
    cav = cav / (np.linalg.norm(cav) + 1e-8)
    sens = float(np.dot(pooled_grads, cav))

    if previous_lstm_mode is not None:
        model.temporal_gate.lstm.train(previous_lstm_mode)

    return sens

## 8. Suppression conceptuelle post-hoc dans l’espace latent

In [32]:
def remove_concept_projection_from_feature_map(feature_map, cav, removal_strength=1.0):
    """
    Retire la composante conceptuelle d'une feature map spatio-temporelle.

    feature_map: Tensor [B,C,T,H,W]
    cav: numpy array [C]

    Cette version est adaptée à notre modèle car les CAVs sont appris dans
    l'espace des features gated [C]. On ne passe donc pas directement par
    model.classifier(z), car le classifieur final reçoit le contexte LSTM et non
    le vecteur [C] brut.
    """
    cav_t = torch.tensor(cav, dtype=feature_map.dtype, device=feature_map.device)
    cav_t = cav_t.view(1, -1, 1, 1, 1)
    cav_t = cav_t / (torch.norm(cav_t, dim=1, keepdim=True) + 1e-8)

    projection = torch.sum(feature_map * cav_t, dim=1, keepdim=True) * cav_t
    removed = feature_map - removal_strength * projection
    inserted = removal_strength * projection

    return removed, inserted


def classify_from_gated_features(model, gated_features):
    """
    Recalcule les logits à partir de features gated modifiées.

    Étapes identiques au forward normal après le backbone:
    gated_features -> pooling spatial -> temporal gate -> classifier.
    """
    frame_features = gated_features.mean(dim=(3, 4)).permute(0, 2, 1)  # [B,T,C]
    context, alpha, beta, seq = model.temporal_gate(frame_features)
    logits = model.classifier(context)
    return logits


def forward_with_concept_removed(model, video, cav, device, removal_strength=1.0):
    """
    Forward robuste pour l'évaluation causale TCAV conditionnée.

    Retourne:
        logits_original : prédiction normale
        logits_removed  : prédiction après suppression de la direction conceptuelle
        logits_inserted : prédiction en gardant uniquement la composante conceptuelle

    Cette fonction corrige l'erreur précédente: forward_with_concept_removed
    manquante/non définie.
    """
    model.eval()
    x = video.unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(x, return_explanations=True)
        logits_original = outputs["logits"]

        if "gated_features" in outputs:
            gated = outputs["gated_features"]
        elif "features" in outputs:
            # fallback si la version du modèle ne retourne pas gated_features
            gated = outputs["features"]
        else:
            raise KeyError("Le modèle doit retourner 'gated_features' ou 'features'.")

        gated_removed, gated_inserted = remove_concept_projection_from_feature_map(
            gated,
            cav,
            removal_strength=removal_strength,
        )

        logits_removed = classify_from_gated_features(model, gated_removed)
        logits_inserted = classify_from_gated_features(model, gated_inserted)

    return logits_original, logits_removed, logits_inserted



## 9. Construction ou chargement de la banque TCAV

In [33]:
def build_tcav_concept_bank_v2(
    checkpoint_path,
    split="train",
    max_samples=800,
    min_pos=80,
    min_neg=80,
    output_path="./xai_outputs_attention_gated_final/tcav_dynamic_concept_bank.pkl"
):
    """
    Construit une banque TCAV avec les concepts dynamiques robustes.
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    model, cfg = load_attention_gated_model_for_tcav(checkpoint_path)

    dataset = UCF101VideoFolderDataset(
        root_dir=split_dir(cfg, split),
        clip_len=cfg.clip_len,
        frame_size=cfg.frame_size,
        train=False,
    )

    print("Calcul des seuils dynamiques...")
    thresholds = compute_dataset_dynamic_thresholds(dataset, max_samples=max_samples)

    activations = []
    concept_labels = {c: [] for c in TCAV_DYNAMIC_CONCEPTS}
    paths = []

    n = min(max_samples, len(dataset))

    print("Extraction des activations et attribution des concepts...")
    for i in tqdm(range(n), desc=f"Scan TCAV {split}"):
        sample = dataset[i]
        video = sample["video"]

        stats = extract_dynamic_statistics(video)
        assigned = assign_dynamic_concepts(stats, thresholds)
        act = extract_tcav_activation(model, video, cfg.device)

        activations.append(act)
        paths.append(sample["path"])

        for c in TCAV_DYNAMIC_CONCEPTS:
            concept_labels[c].append(assigned[c])

    X = np.stack(activations).astype(np.float32)

    concept_bank = {
        "concepts": {},
        "thresholds": thresholds,
        "concept_descriptions": CONCEPT_DESCRIPTIONS,
        "paths": paths,
        "feature_dim": X.shape[1],
        "removed_concepts": ["mouvement_fort", "mouvement_faible", "mouvement_soutenu"],
    }

    for concept in TCAV_DYNAMIC_CONCEPTS:
        y = np.array(concept_labels[concept], dtype=np.int64)

        pos = int(y.sum())
        neg = int(len(y) - pos)

        if pos < min_pos or neg < min_neg:
            print(f"Concept {concept}: rejeté, pos={pos}, neg={neg}. Pas assez d'exemples.")
            continue

        clf = LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="liblinear",
        )
        clf.fit(X, y)

        pred = clf.predict(X)
        probe_acc = accuracy_score(y, pred)

        cav = clf.coef_[0].astype(np.float32)
        cav = cav / (np.linalg.norm(cav) + 1e-8)

        concept_bank["concepts"][concept] = {
            "cav": cav,
            "probe_acc": float(probe_acc),
            "pos": pos,
            "neg": neg,
            "description": CONCEPT_DESCRIPTIONS[concept],
        }

        print(f"Concept {concept}: probe_acc={probe_acc:.3f}, pos={pos}, neg={neg}")

    with open(output_path, "wb") as f:
        pickle.dump(concept_bank, f)

    print("Banque TCAV sauvegardée:", output_path)
    return output_path

In [ ]:
# Construction ou chargement de la banque TCAV

# construire une nouvelle banque TCAV.
# tcav_bank_path = build_tcav_concept_bank_v2(
#     checkpoint_path=checkpoint_path,
#     split="val",
#     max_samples=1673,
#     min_pos=80,
#     min_neg=80,
#     output_path="./xai_outputs_attention_gated_final/tcav_dynamic_concept_bank.pkl",
# )


In [34]:
# si la banque existe déjà
tcav_bank_path = "/kaggle/input/datasets/younessouarda/tcav-dynamic-concept-bank/tcav_dynamic_concept_bank.pkl"

print("Banque TCAV utilisée:", tcav_bank_path)


Banque TCAV utilisée: /kaggle/input/datasets/younessouarda/tcav-dynamic-concept-bank/tcav_dynamic_concept_bank.pkl


## 10. Scores TCAV pour une vidéo

In [35]:
def compute_tcav_concept_scores(model, video, tcav_bank_path, device, class_idx=None):
    """
    Calcule les scores TCAV pour une vidéo.

    score_presence:
        présence approximative du concept dans l'activation latente.

    sensitivity:
        influence directionnelle du concept sur la prédiction.
    """
    with open(tcav_bank_path, "rb") as f:
        concept_bank = pickle.load(f)

    act = extract_tcav_activation(model, video, device)

    scores = {}

    for concept, data in concept_bank["concepts"].items():
        cav = data["cav"]
        raw_score = float(np.dot(act, cav))
        presence_score = float(1.0 / (1.0 + np.exp(-raw_score)))

        try:
            sensitivity = extract_tcav_gradient_sensitivity(
                model=model,
                video=video,
                cav=cav,
                device=device,
                class_idx=class_idx,
            )
        except Exception as e:
            sensitivity = np.nan
            print(f"Attention: sensibilité TCAV non calculée pour {concept}: {e}")

        scores[concept] = {
            "score": presence_score,
            "raw_score": raw_score,
            "sensitivity": float(sensitivity) if not np.isnan(sensitivity) else np.nan,
            "probe_acc": data["probe_acc"],
            "pos": data["pos"],
            "neg": data["neg"],
            "description": data["description"],
        }

    return scores


def interpret_tcav_scores(tcav_scores, min_score=0.55, min_probe_acc=0.80, top_k=3):
    """
    Construit une interprétation conceptuelle plus souple et plus réaliste.

    Règle principale:
        - concept dominant si score >= 0.55 et probe_acc >= 0.80.

    Si aucun concept ne dépasse ce seuil, on affiche quand même les concepts
    les plus présents afin d'éviter une explication vide, surtout pour des
    actions à mouvement faible ou très localisé comme ApplyEyeMakeup.
    """
    candidates = []

    for concept, data in tcav_scores.items():
        candidates.append((
            concept,
            float(data["score"]),
            float(data.get("sensitivity", np.nan)),
            float(data["probe_acc"])
        ))

    candidates = sorted(candidates, key=lambda x: x[1], reverse=True)

    dominant = [
        item for item in candidates
        if item[1] >= min_score and item[3] >= min_probe_acc
    ]

    def format_item(concept, score, sens, probe_acc):
        if np.isnan(sens):
            direction = "influence non calculée"
        elif sens > 0:
            direction = "contribue positivement à la décision"
        else:
            direction = "présent mais influence négative/faible sur la décision"
        return f"{concept} (score={score:.2f}, probe={probe_acc:.2f}, {direction})"

    if len(dominant) > 0:
        parts = [format_item(*item) for item in dominant[:top_k]]
        return "Concepts dynamiques dominants: " + "; ".join(parts)

    # Explication de repli: top-k concepts les plus présents, même sans seuil dominant.
    top_present = candidates[:top_k]
    parts = [format_item(*item) for item in top_present]
    return (
        "Aucun concept ne dépasse le seuil dominant strict. "
        "Concepts les plus présents: " + "; ".join(parts)
    )

## 11. Explication complète d’une vidéo avec TCAV

In [36]:
def explain_one_sample_with_tcav_v2(
    checkpoint_path,
    tcav_bank_path,
    sample_index=0,
    split="test"
):
    """
    Génère une explication complète:
        - prédiction ;
        - Grad-CAM spatial ;
        - attention temporelle ;
        - dynamique ;
        - concepts TCAV.
    """
    report, sample = explain_one_sample(
        checkpoint_path=checkpoint_path,
        sample_index=sample_index,
        split=split,
    )

    model, cfg = load_attention_gated_model_for_tcav(checkpoint_path)

    class_idx = None
    if "prediction_index" in report:
        class_idx = int(report["prediction_index"])

    tcav_scores = compute_tcav_concept_scores(
        model=model,
        video=sample["video"],
        tcav_bank_path=tcav_bank_path,
        device=cfg.device,
        class_idx=class_idx,
    )

    sorted_scores = dict(
        sorted(tcav_scores.items(), key=lambda x: x[1]["score"], reverse=True)
    )

    report["tcav_dynamic_concepts"] = sorted_scores
    report["tcav_concept_explanation"] = interpret_tcav_scores(sorted_scores)

    print("Explication conceptuelle TCAV:")
    print(report["tcav_concept_explanation"])

    return report, sample



In [37]:
def show_tcav_concepts(report):
    if "tcav_dynamic_concepts" not in report:
        print("Aucun score TCAV dans le rapport.")
        return

    rows = ""

    for concept, data in report["tcav_dynamic_concepts"].items():
        score = data["score"]
        probe_acc = data["probe_acc"]
        sensitivity = data.get("sensitivity", np.nan)
        desc = data["description"]

        if score >= 0.80:
            niveau = "fort"
        elif score >= 0.60:
            niveau = "modéré"
        else:
            niveau = "faible"

        if np.isnan(sensitivity):
            sens_txt = "N/A"
        else:
            sens_txt = f"{sensitivity:.4f}"

        rows += f"""
        <tr>
            <td>{concept}</td>
            <td>{score:.3f}</td>
            <td>{sens_txt}</td>
            <td>{probe_acc:.3f}</td>
            <td>{niveau}</td>
            <td>{desc}</td>
        </tr>
        """

    html = f"""
    <div style="font-family: Arial; max-width: 1000px;">
        <h3>Analyse conceptuelle TCAV dynamique</h3>
        <table border="1" style="border-collapse: collapse; font-size: 14px; width: 100%;">
            <tr>
                <th>Concept</th>
                <th>Score TCAV</th>
                <th>Sensibilité</th>
                <th>Probe accuracy</th>
                <th>Niveau</th>
                <th>Description</th>
            </tr>
            {rows}
        </table>
        <p><b>Interprétation :</b> {report.get("tcav_concept_explanation", "")}</p>
    </div>
    """

    display(HTML(html))


def show_original_and_explanation_with_tcav(report, sample, width=750):
    show_original_and_explanation(report, sample, width=width)
    show_tcav_concepts(report)

## 12. Évaluation TCAV conditionnée par classes pertinentes

Chaque concept est évalué uniquement sur les classes UCF101 où il est sémantiquement pertinent. Cela évite de diluer les scores en évaluant un concept sur des classes où il n’est pas censé intervenir.

In [39]:
CONCEPT_TO_RELEVANT_CLASSES = {
    "mouvement_brusque": [
        "BoxingPunchingBag", "BoxingSpeedBag", "Punch",
        "TennisSwing", "BaseballPitch", "CricketShot",
        "JavelinThrow", "HammerThrow", "Shotput",
        "BasketballDunk", "VolleyballSpiking"
    ],

    "changement_rapide": [
        "JavelinThrow", "BaseballPitch", "TennisSwing",
        "HammerThrow", "Shotput", "HighJump",
        "LongJump", "PoleVault", "Diving",
        "BasketballDunk", "VolleyballSpiking"
    ],

    "mouvement_intermitent": [
        "SoccerJuggling", "YoYo", "JugglingBalls",
        "JumpRope", "Drumming", "PlayingDaf",
        "PlayingTabla", "Billiards"
    ],

    "mouvement_accelere": [
        "JavelinThrow", "HighJump", "LongJump",
        "PoleVault", "BasketballDunk", "Diving",
        "BaseballPitch", "TennisSwing", "VolleyballSpiking"
    ],

    "mouvement_periodique": [
        "JumpRope", "Biking", "Rowing",
        "WalkingWithDog", "HorseRiding", "TaiChi",
        "HulaHoop", "SalsaSpin", "ApplyEyeMakeup"
    ],

    "mouvement_localise": [
        "ApplyEyeMakeup", "ApplyLipstick", "Typing",
        "Knitting", "PlayingPiano", "WritingOnBoard",
        "BrushingTeeth", "CuttingInKitchen", "ShavingBeard"
    ],

    "mouvement_global": [
        "Biking", "HorseRiding", "SkateBoarding",
        "Surfing", "Skiing", "WalkingWithDog",
        "Basketball", "SoccerJuggling", "FloorGymnastics",
        "FrontCrawl", "PlayingBasketball"
    ],

    "mouvement_fin_precision": [
        "ApplyEyeMakeup", "ApplyLipstick", "Typing",
        "Knitting", "PlayingPiano", "WritingOnBoard",
        "PlayingGuitar", "BrushingTeeth", "CuttingInKitchen",
        "ShavingBeard"
    ],
}

print("Concepts conditionnés définis:")
for concept, classes in CONCEPT_TO_RELEVANT_CLASSES.items():
    print(f"- {concept}: {len(classes)} classes candidates")

Concepts conditionnés définis:
- mouvement_brusque: 11 classes candidates
- changement_rapide: 11 classes candidates
- mouvement_intermitent: 8 classes candidates
- mouvement_accelere: 9 classes candidates
- mouvement_periodique: 9 classes candidates
- mouvement_localise: 9 classes candidates
- mouvement_global: 11 classes candidates
- mouvement_fin_precision: 10 classes candidates


In [41]:

# TCAV SCORE — Présence conceptuelle depuis le latent

def concept_presence_from_latent(model, video, cav, device):
    """
    Calcule le score de présence TCAV d'un concept pour une vidéo.

    Utilise exactement le même pooling latent que la banque TCAV:
        feats -> moyenne spatiale -> max temporel

    Retour:
        tcav_score dans [0,1]
        raw_score non normalisé
    """
    model.eval()

    with torch.no_grad():
        z = extract_tcav_activation(
            model=model,
            video=video,
            device=device
        )

    cav_norm = cav / (np.linalg.norm(cav) + 1e-8)

    raw_score = float(np.dot(z, cav_norm))
    tcav_score = float(1.0 / (1.0 + np.exp(-raw_score)))

    return tcav_score, raw_score

In [42]:
def get_dataset_class_index_map(dataset):
    """
    Construit un dictionnaire classe -> indices vidéos pour le dataset chargé.
    """
    class_to_indices = {}

    for idx, sample_info in enumerate(dataset.samples):
        # Selon la version du dataset, sample_info peut contenir path,label,class_name.
        if isinstance(sample_info, (list, tuple)) and len(sample_info) >= 3:
            class_name = sample_info[2]
        else:
            # fallback robuste
            try:
                class_name = dataset[idx]["class_name"]
            except Exception:
                path = sample_info[0] if isinstance(sample_info, (list, tuple)) else str(sample_info)
                class_name = os.path.basename(os.path.dirname(path))

        class_to_indices.setdefault(class_name, []).append(idx)

    return class_to_indices


def get_relevant_indices_for_concept(dataset, concept, concept_to_classes):
    """
    Retourne les indices des vidéos appartenant aux classes pertinentes pour un concept.
    Les classes non présentes dans le dataset sont ignorées.
    """
    class_to_indices = get_dataset_class_index_map(dataset)
    available_classes = set(class_to_indices.keys())

    requested_classes = concept_to_classes.get(concept, [])
    valid_classes = [c for c in requested_classes if c in available_classes]
    missing_classes = [c for c in requested_classes if c not in available_classes]

    indices = []
    for c in valid_classes:
        indices.extend(class_to_indices[c])

    return indices, valid_classes, missing_classes


def evaluate_concept_fidelity_conditioned_by_classes(
    checkpoint_path,
    tcav_bank_path,
    split="test",
    concept_to_classes=None,
    max_samples_per_concept=None,
    output_csv="./xai_outputs_attention_gated_final/tcav_concept_fidelity_conditioned.csv",
    removal_strength=1.0,
    random_seed=42,
):
    """
    Évalue la fidélité conceptuelle uniquement sur les classes pertinentes à chaque concept.

    Pour chaque concept:
        1. On sélectionne les vidéos dont la classe appartient à la liste pertinente.
        2. On retire la direction conceptuelle CAV dans les features gated.
        3. On mesure la chute de confiance de la prédiction.

    Cette évaluation est plus juste que l'évaluation globale, car elle ne pénalise pas un concept
    lorsqu'il est naturellement absent d'une classe non pertinente.
    """
    if concept_to_classes is None:
        concept_to_classes = CONCEPT_TO_RELEVANT_CLASSES

    os.makedirs(os.path.dirname(output_csv), exist_ok=True)

    model, cfg = load_attention_gated_model_for_tcav(checkpoint_path)

    dataset = UCF101VideoFolderDataset(
        root_dir=split_dir(cfg, split),
        clip_len=cfg.clip_len,
        frame_size=cfg.frame_size,
    )

    with open(tcav_bank_path, "rb") as f:
        concept_bank = pickle.load(f)

    rng = np.random.default_rng(random_seed)
    rows = []

    for concept, data in concept_bank["concepts"].items():
        if concept not in concept_to_classes:
            print(f"Concept ignoré car absent du mapping: {concept}")
            continue

        indices, valid_classes, missing_classes = get_relevant_indices_for_concept(
            dataset,
            concept,
            concept_to_classes,
        )

        if len(indices) == 0:
            print(f"Concept {concept}: aucune vidéo pertinente trouvée dans le split {split}.")
            continue

        if max_samples_per_concept is not None and len(indices) > max_samples_per_concept:
            indices = list(rng.choice(indices, size=max_samples_per_concept, replace=False))

        print(
            f"Concept {concept}: {len(indices)} vidéos évaluées | "
            f"classes valides={len(valid_classes)} | classes manquantes={len(missing_classes)}"
        )

        cav = data["cav"]
        probe_acc = float(data.get("probe_acc", np.nan))

        for idx in tqdm(indices, desc=f"TCAV conditionné - {concept}"):
            sample = dataset[idx]
            video = sample["video"]

            # Prédiction originale.
            with torch.no_grad():
                x = video.unsqueeze(0).to(cfg.device)
                logits = model(x, return_explanations=False)
                probs = F.softmax(logits, dim=1)
                pred_idx = int(probs.argmax(dim=1).item())
                pred_conf = float(probs[0, pred_idx].detach().cpu())

            try:
                logits_original, logits_removed, logits_inserted = forward_with_concept_removed(
                    model=model,
                    video=video,
                    cav=cav,
                    device=cfg.device,
                    removal_strength=removal_strength,
                )

                probs_removed = F.softmax(logits_removed, dim=1)
                probs_inserted = F.softmax(logits_inserted, dim=1)

                conf_removed = float(probs_removed[0, pred_idx].detach().cpu())
                conf_inserted = float(probs_inserted[0, pred_idx].detach().cpu())
                deletion_drop = pred_conf - conf_removed
                insertion_gap = pred_conf - conf_inserted

            except Exception as e:
                conf_removed = np.nan
                conf_inserted = np.nan
                deletion_drop = np.nan
                insertion_gap = np.nan
                print(f"Erreur concept deletion pour {concept}, vidéo {idx}: {e}")

            try:
                tcav_score, raw_score = concept_presence_from_latent(
                    model=model,
                    video=video,
                    cav=cav,
                    device=cfg.device,
                )
            except Exception:
                tcav_score, raw_score = np.nan, np.nan

            try:
                sensitivity = extract_tcav_gradient_sensitivity(
                    model=model,
                    video=video,
                    cav=cav,
                    device=cfg.device,
                    class_idx=pred_idx,
                )
            except Exception:
                sensitivity = np.nan

            rows.append({
                "index": idx,
                "video_path": sample["path"],
                "ground_truth": sample["class_name"],
                "prediction_index": pred_idx,
                "prediction_confidence": pred_conf,
                "concept": concept,
                "relevant_classes": ",".join(valid_classes),
                "num_relevant_classes": len(valid_classes),
                "tcav_score": tcav_score,
                "tcav_raw_score": raw_score,
                "tcav_sensitivity": sensitivity,
                "probe_acc": probe_acc,
                "concept_confidence_after_removal": conf_removed,
                "concept_deletion_drop": deletion_drop,
                "concept_insertion_confidence": conf_inserted,
                "concept_insertion_gap_lower_is_better": insertion_gap,
                "conditioned_evaluation": True,
            })

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print("Évaluation TCAV conditionnée sauvegardée:", output_csv)
    return df

## 13. Résumé et interprétation des résultats

In [43]:
def summarize_conditioned_concept_fidelity(df_conditioned):
    """
    Résume la fidélité conceptuelle conditionnée par classes pertinentes.
    """
    summary = df_conditioned.groupby("concept").agg(
        n_videos=("concept", "count"),
        n_classes=("num_relevant_classes", "max"),
        mean_tcav_score=("tcav_score", "mean"),
        median_tcav_score=("tcav_score", "median"),
        mean_probe_acc=("probe_acc", "mean"),
        mean_sensitivity=("tcav_sensitivity", "mean"),
        positive_sensitivity_rate=("tcav_sensitivity", lambda x: float(np.mean(np.array(x) > 0))),
        mean_deletion_drop=("concept_deletion_drop", "mean"),
        median_deletion_drop=("concept_deletion_drop", "median"),
        positive_drop_rate=("concept_deletion_drop", lambda x: float(np.mean(np.array(x) > 0))),
        strong_drop_rate_0_02=("concept_deletion_drop", lambda x: float(np.mean(np.array(x) > 0.02))),
        strong_drop_rate_0_05=("concept_deletion_drop", lambda x: float(np.mean(np.array(x) > 0.05))),
        mean_insertion_confidence=("concept_insertion_confidence", "mean"),
        mean_insertion_gap=("concept_insertion_gap_lower_is_better", "mean"),
    ).reset_index()

    summary["conditioned_concept_score"] = (
        0.25 * summary["mean_probe_acc"].fillna(0)
        + 0.25 * summary["mean_tcav_score"].fillna(0)
        + 0.25 * summary["positive_drop_rate"].fillna(0)
        + 0.25 * summary["positive_sensitivity_rate"].fillna(0)
    )

    summary = summary.sort_values("conditioned_concept_score", ascending=False)
    display(summary)
    return summary


def interpret_conditioned_concept_summary(summary_df):
    """
    Interprète les résultats conditionnés par classes pertinentes.
    """
    print("Interprétation TCAV conditionnée par classes pertinentes")
    print("-------------------------------------------------------")

    for _, row in summary_df.iterrows():
        concept = row["concept"]
        probe = row["mean_probe_acc"]
        score = row["mean_tcav_score"]
        drop = row["mean_deletion_drop"]
        pos_drop = row["positive_drop_rate"]
        sens_rate = row["positive_sensitivity_rate"]
        n = int(row["n_videos"])
        n_cls = int(row["n_classes"])

        if probe >= 0.80 and score >= 0.55 and pos_drop >= 0.55 and sens_rate >= 0.55:
            status = "concept pertinent et partiellement causal sur ses classes"
        elif probe >= 0.80 and score >= 0.55 and (pos_drop >= 0.50 or sens_rate >= 0.50):
            status = "concept pertinent mais causalité modérée"
        elif probe >= 0.80 and score >= 0.55:
            status = "concept présent mais causalité faible"
        else:
            status = "concept à surveiller"

        print(
            f"{concept}: {status} | "
            f"n={n}, classes={n_cls}, probe={probe:.3f}, "
            f"tcav={score:.3f}, drop={drop:.4f}, "
            f"positive_drop={pos_drop:.3f}, positive_sens={sens_rate:.3f}"
        )

In [44]:
# Évaluation TCAV conditionnée par classes pertinentes

df_tcav_conditioned = evaluate_concept_fidelity_conditioned_by_classes(
    checkpoint_path=checkpoint_path,
    tcav_bank_path=tcav_bank_path,
    split="test",
    concept_to_classes=CONCEPT_TO_RELEVANT_CLASSES,
    max_samples_per_concept=300,
    output_csv="./xai_outputs_attention_gated_final/tcav_conditioned_results.csv",
    removal_strength=1.0,
)

summary_tcav_conditioned = summarize_conditioned_concept_fidelity(df_tcav_conditioned)
interpret_conditioned_concept_summary(summary_tcav_conditioned)


Concept mouvement_brusque: 203 vidéos évaluées | classes valides=11 | classes manquantes=0


TCAV conditionné - mouvement_brusque: 100%|██████████| 203/203 [00:54<00:00,  3.74it/s]


Concept changement_rapide: 195 vidéos évaluées | classes valides=11 | classes manquantes=0


TCAV conditionné - changement_rapide: 100%|██████████| 195/195 [00:48<00:00,  4.01it/s]


Concept mouvement_intermitent: 142 vidéos évaluées | classes valides=8 | classes manquantes=0


...

Concept mouvement_fin_precision: 172 vidéos évaluées | classes valides=10 | classes manquantes=0


TCAV conditionné - mouvement_fin_precision: 100%|██████████| 172/172 [00:44<00:00,  3.84it/s]

Évaluation TCAV conditionnée sauvegardée: ./xai_outputs_attention_gated_final/tcav_conditioned_results.csv


,concept,n_videos,n_classes,mean_tcav_score,median_tcav_score,mean_probe_acc,mean_sensitivity,positive_sensitivity_rate,mean_deletion_drop,median_deletion_drop,positive_drop_rate,strong_drop_rate_0_02,strong_drop_rate_0_05,mean_insertion_confidence,mean_insertion_gap,conditioned_concept_score
3,mouvement_fin_precision,172,10,0.477598,0.482747,0.930066,0.000977,0.767442,0.000535,-1.788139e-07,0.447674,0.017442,0.0,0.019327,0.937628,0.655695
1,mouvement_accelere,158,9,0.492599,0.497237,0.903766,0.000157,0.734177,0.000072,0.000000e+00,0.430380,0.000000,0.0,0.004207,0.917006,0.640230
6,mouvement_localise,152,9,0.479390,0.485321,0.926479,0.000455,0.532895,0.000313,4.768372e-07,0.572368,0.019737,0.0,0.018546,0.933770,0.627783
0,changement_rapide,195,11,0.492197,0.497909,0.906754,-0.000016,0.543590,-0.000265,0.000000e+00,0.456410,0.000000,0.0,0.006531,0.916526,0.599738
4,mouvement_global,189,10,0.493922,0.498183,0.875075,-0.000071,0.576720,-0.000014,0.000000e+00,0.433862,0.000000,0.0,0.013785,0.906365,0.594895
2,mouvement_brusque,203,11,0.476010,0.479289,0.876270,-0.000243,0.408867,-0.000253,0.000000e+00,0.413793,0.000000,0.0,0.006593,0.922946,0.543735
5,mouvement_intermitent,142,8,0.470428,0.473690,0.893007,-0.000428,0.330986,0.000070,0.000000e+00,0.443662,0.000000,0.0,0.004550,0.958974,0.534521
7,mouvement_periodique,155,9,0.448555,0.454363,0.883443,-0.000233,0.303226,0.000106,0.000000e+00,0.470968,0.000000,0.0,0.005107,0.943970,0.526548


Interprétation TCAV conditionnée par classes pertinentes
-------------------------------------------------------
mouvement_fin_precision: concept à surveiller | n=172, classes=10, probe=0.930, tcav=0.478, drop=0.0005, positive_drop=0.448, positive_sens=0.767
mouvement_accelere: concept à surveiller | n=158, classes=9, probe=0.904, tcav=0.493, drop=0.0001, positive_drop=0.430, positive_sens=0.734
mouvement_localise: concept à surveiller | n=152, classes=9, probe=0.926, tcav=0.479, drop=0.0003, positive_drop=0.572, positive_sens=0.533
changement_rapide: concept à surveiller | n=195, classes=11, probe=0.907, tcav=0.492, drop=-0.0003, positive_drop=0.456, positive_sens=0.544
mouvement_global: concept à surveiller | n=189, classes=10, probe=0.875, tcav=0.494, drop=-0.0000, positive_drop=0.434, positive_sens=0.577
mouvement_brusque: concept à surveiller | n=203, classes=11, probe=0.876, tcav=0.476, drop=-0.0003, positive_drop=0.414, positive_sens=0.409
mouvement_intermitent: concept à survei